In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import ast
import warnings
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from transformers import EarlyStoppingCallback

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

start_path = '/content/drive/MyDrive/TaskB/TaskB'
new_model_path = os.path.join(start_path, 'bi_encoder_mpnet_v1')

with open(os.path.join(start_path, 'training', 'jobid2terms.json'), 'r', encoding='utf-8') as f:
    job_dict = json.load(f)
with open(os.path.join(start_path, 'training', 'skillid2terms.json'), 'r', encoding='utf-8') as f:
    skill_dict = json.load(f)
job2skill_df = pd.read_csv(os.path.join(start_path, 'training', 'job2skill.tsv'), sep='\t', header=None, names=['job_id', 'skill_id', 'essential']).dropna()

anchors = []
positives = []
for _, row in job2skill_df.iterrows():
    job_text = " ".join(job_dict.get(str(row['job_id']), []))
    job_text = " ".join(job_text.split()[:80]) if job_text.strip() else "unknown"
    correct_skill_id = str(row['skill_id'])
    correct_skill_text = " ".join(skill_dict.get(correct_skill_id, []))
    correct_skill_text = " ".join(correct_skill_text.split()[:80]) if correct_skill_text.strip() else "unknown"

    if job_text != "unknown" and correct_skill_text != "unknown":
        anchors.append(job_text)
        positives.append(correct_skill_text)

train_dataset = Dataset.from_dict({"anchor": anchors, "positive": positives})

queries_df = pd.read_csv(os.path.join(start_path, 'validation', 'queries'), sep='\t').dropna()
corpus_df = pd.read_csv(os.path.join(start_path, 'validation', 'corpus_elements'), sep='\t').dropna(subset=['c_id'])
queries_val = {str(row['q_id']): " ".join(str(row['jobtitle']).split()[:80]) if pd.notna(row['jobtitle']) else "unknown" for _, row in queries_df.iterrows()}

corpus_val = {}
for _, row in corpus_df.iterrows():
    cid = str(row['c_id'])
    try:
        aliases = ast.literal_eval(row['skill_aliases'])
        text = " ".join(" ".join(aliases).split()[:80])
    except:
        text = " ".join(str(row['skill_aliases']).split()[:80])
    corpus_val[cid] = text if text.strip() else "unknown"

qrels_df = pd.read_csv(os.path.join(start_path, 'validation', 'qrels.tsv'), sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance']).dropna()

queries_eval = {}
relevant_docs = {}

for qid, query_text in queries_val.items():
    pos_cids = qrels_df[(qrels_df['q_id'].astype(str) == qid) & (qrels_df['relevance'] > 0)]['c_id'].astype(str).tolist()
    if pos_cids:
        queries_eval[qid] = query_text
        relevant_docs[qid] = set(pos_cids)

evaluator = InformationRetrievalEvaluator(queries_eval, corpus_val, relevant_docs, name='val_ir')

model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
train_loss = MultipleNegativesRankingLoss(model)

batch_size = 32
eval_steps_count = max(1, len(train_dataset) // (batch_size * 30))

args = SentenceTransformerTrainingArguments(
    output_dir=new_model_path,
    num_train_epochs=8,
    per_device_train_batch_size=batch_size,
    eval_strategy="steps",
    eval_steps=eval_steps_count,
    save_strategy="steps",
    save_steps=eval_steps_count,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@100",
    warmup_ratio=0.1,
    dataloader_num_workers=2,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    evaluator=evaluator,
    loss=train_loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=90)]
)

trainer.train(resume_from_checkpoint=True)

best_model = SentenceTransformer(new_model_path)

cids = list(corpus_val.keys())
corpus_texts = [corpus_val[cid] for cid in cids]
corpus_embeddings = best_model.encode(corpus_texts, batch_size=128, convert_to_tensor=True, show_progress_bar=False)

results = []
for qid, query_text in queries_val.items():
    query_embedding = best_model.encode(query_text, convert_to_tensor=True, show_progress_bar=False)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=len(cids))[0]

    for rank, hit in enumerate(hits):
        cid = cids[hit['corpus_id']]
        score = hit['score']
        results.append(f"{qid}\tQ0\t{cid}\t{rank + 1}\t{float(score)}\tbiencoder_run")

with open(os.path.join(start_path, 'predictions.trec'), 'w') as f:
    f.write("\n".join(results))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'train_runtime': '2627', 'train_samples_per_second': '349.4', 'train_steps_per_second': '10.92', 'train_loss': '0.02676', 'epoch': '8'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
pip install pytrec_eval

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309353 sha256=d918f52e03d340f1c999d34d3082ffc93e597cff538abb6b679687e91555d9bc
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [ ]:
import os
import pytrec_eval
import pandas as pd

start_path = '/content/drive/MyDrive/TaskB/TaskB'
qrels_path = os.path.join(start_path, 'validation', 'qrels.tsv')
trec_path = os.path.join(start_path, 'predictions.trec')

qrels_df = pd.read_csv(qrels_path, sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance'])
qrels = {}
for _, row in qrels_df.iterrows():
    qid = str(row['q_id'])
    if qid not in qrels:
        qrels[qid] = {}
    qrels[qid][str(row['c_id'])] = int(row['relevance'])

run = {}
with open(trec_path, 'r') as f:
    for line in f:
        qid, _, docid, rank, score, _ = line.strip().split()
        if qid not in run:
            run[qid] = {}
        run[qid][docid] = float(score)

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10'})
results = evaluator.evaluate(run)

ndcg_scores = [query_measures['ndcg_cut_10'] for query_measures in results.values()]
average_ndcg = sum(ndcg_scores) / len(ndcg_scores)
print(average_ndcg)

0.3135201676258233


In [ ]:
!git clone https://github.com/TalentCLEF/talentclef26_evaluation_script.git
!pip install -r /content/talentclef26_evaluation_script/requirements.txt

Cloning into 'talentclef26_evaluation_script'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 64 (delta 7), reused 60 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 29.46 KiB | 1.18 MiB/s, done.
Resolving deltas: 100% (7/7), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.4/285.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.8 MB/s eta 0:00:00
  Created 

In [ ]:
import os
import subprocess

start_path = '/content/drive/MyDrive/TaskB/TaskB'
old_trec_path = os.path.join(start_path, 'predictions.trec')
new_trec_path = os.path.join(start_path, 'predictions_formatted.trec')

with open(old_trec_path, 'r') as infile, open(new_trec_path, 'w') as outfile:
    for line in infile:
        outfile.write(line.replace('\t', ' '))

qrels_file = os.path.join(start_path, 'validation', 'qrels.tsv')

command = [
    "python",
    "/content/talentclef26_evaluation_script/talentclef_evaluate.py",
    "--task", "B",
    "--qrels", qrels_file,
    "--run", new_trec_path
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

TalentCLEF 2026 - Task B Evaluation
Received parameters:
  Task: B
  Qrels: /content/drive/MyDrive/TaskB/TaskB/validation/qrels.tsv
  Run: /content/drive/MyDrive/TaskB/TaskB/predictions_formatted.trec

Loading qrels...
Loading run...

Running Task B evaluation...

EVALUATION RESULTS

--- General Relevance (1 or 2 → relevant) ---
ndcg: 0.7050
map: 0.2025
mrr: 0.6461
precision@5: 0.4224
precision@10: 0.4030
precision@100: 0.2972

--- Graded Relevance (2 = core skill, 1 = contextual skill) ---
ndcg: 0.6595
map: 0.2025
mrr: 0.6461
precision@5: 0.4224
precision@10: 0.4030
precision@100: 0.2972



In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission.zip /content/drive/MyDrive/TaskB/TaskB/predictions_formatted.trec

  adding: predictions_formatted.trec (deflated 79%)


In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
import os

start_path = '/content/drive/MyDrive/TaskB/TaskB'
# This is the formatted file you generated a few steps ago
old_trec_path = os.path.join(start_path, 'predictions_formatted.trec')
# New file following the run_xx-xx_teamname.trec rule
new_trec_path = os.path.join(start_path, 'run_en-en_Olive.trec')

shutil.copy(old_trec_path, new_trec_path)

'/content/drive/MyDrive/TaskB/TaskB/run_en-en_Olive.trec'

In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip /content/drive/MyDrive/TaskB/TaskB/run_en-en_Olive.trec

  adding: run_en-en_Olive.trec (deflated 79%)


In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_test_Olive.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TaskB/TaskB/training/job2skill.tsv',
                 sep='\t', header=None, names=['job_id','skill_id','essential'])
print(f"Total pairs : {len(df)}")
print(f"Essential   : {len(df[df.essential=='essential'])}")
print(f"Optional    : {len(df[df.essential=='optional'])}")

Total pairs : 114699
Essential   : 62480
Optional    : 52219
